In [1]:
import pandas as pd

# Read dataset
df = pd.read_csv("Dataset_Day15.csv", header=None)

# Display first few rows
print(df.head())

# Replace empty strings with NaN
df.replace('', pd.NA, inplace=True)

# Remove null values
df.dropna(how='all', inplace=True)

# Shape after cleaning
print("Shape =", df.shape)

              0          1           2                 3             4   \
0         shrimp    almonds     avocado    vegetables mix  green grapes   
1        burgers  meatballs        eggs               NaN           NaN   
2        chutney        NaN         NaN               NaN           NaN   
3         turkey    avocado         NaN               NaN           NaN   
4  mineral water       milk  energy bar  whole wheat rice     green tea   

                 5     6               7             8             9   \
0  whole weat flour  yams  cottage cheese  energy drink  tomato juice   
1               NaN   NaN             NaN           NaN           NaN   
2               NaN   NaN             NaN           NaN           NaN   
3               NaN   NaN             NaN           NaN           NaN   
4               NaN   NaN             NaN           NaN           NaN   

               10         11     12     13             14      15  \
0  low fat yogurt  green tea  honey  sala

In [2]:
from mlxtend.preprocessing import TransactionEncoder

# Convert dataframe rows into transaction lists
transactions = []

for i in range(len(df)):
    
    # Remove NaN values from each row
    row = df.iloc[i].dropna().tolist()

    transactions.append(row)

# Apply TransactionEncoder
encoder = TransactionEncoder()

encoded_array = encoder.fit(transactions).transform(transactions)

# Convert to dataframe
encoded_df = pd.DataFrame(encoded_array,
                          columns=encoder.columns_)

print(encoded_df.head())

    asparagus  almonds  antioxydant juice  asparagus  avocado  babies food  \
0       False     True               True      False     True        False   
1       False    False              False      False    False        False   
2       False    False              False      False    False        False   
3       False    False              False      False     True        False   
4       False    False              False      False    False        False   

   bacon  barbecue sauce  black tea  blueberries  ...  turkey  vegetables mix  \
0  False           False      False        False  ...   False            True   
1  False           False      False        False  ...   False           False   
2  False           False      False        False  ...   False           False   
3  False           False      False        False  ...    True           False   
4  False           False      False        False  ...   False           False   

   water spray  white wine  whole weat flour

In [3]:
from mlxtend.frequent_patterns import apriori

# Find frequent itemsets
frequent_itemsets = apriori(encoded_df,
                            min_support=0.02,
                            use_colnames=True)

# Display first few itemsets
print(frequent_itemsets.head())

# Number of itemsets found
print("Total frequent itemsets =",
      len(frequent_itemsets))

    support               itemsets
0  0.020397   frozenset({almonds})
1  0.033329   frozenset({avocado})
2  0.033729  frozenset({brownies})
3  0.087188   frozenset({burgers})
4  0.030129    frozenset({butter})
Total frequent itemsets = 103


In [4]:
from mlxtend.frequent_patterns import association_rules

# Generate association rules
rules = association_rules(frequent_itemsets,
                          metric='confidence',
                          min_threshold=0.15)

# Display first few rules
print(rules.head())

            antecedents                 consequents  antecedent support  \
0     frozenset({eggs})        frozenset({burgers})            0.179709   
1  frozenset({burgers})           frozenset({eggs})            0.087188   
2  frozenset({burgers})   frozenset({french fries})            0.087188   
3  frozenset({burgers})  frozenset({mineral water})            0.087188   
4  frozenset({burgers})      frozenset({spaghetti})            0.087188   

   consequent support   support  confidence      lift  representativity  \
0            0.087188  0.028796    0.160237  1.837830               1.0   
1            0.179709  0.028796    0.330275  1.837830               1.0   
2            0.170911  0.021997    0.252294  1.476173               1.0   
3            0.238368  0.024397    0.279817  1.173883               1.0   
4            0.174110  0.021464    0.246177  1.413918               1.0   

   leverage  conviction  zhangs_metric   jaccard  certainty  kulczynski  
0  0.013128    1.086988 

In [5]:
# Sort rules according to leverage
top_leverage = rules.sort_values(
                    by='leverage',
                    ascending=False)

# Top 5 rules
print(top_leverage[
    ['antecedents',
     'consequents',
     'support',
     'confidence',
     'lift',
     'leverage']
].head())

                   antecedents                 consequents   support  \
53      frozenset({spaghetti})    frozenset({ground meat})  0.039195   
52    frozenset({ground meat})      frozenset({spaghetti})  0.039195   
64  frozenset({mineral water})      frozenset({spaghetti})  0.059725   
63      frozenset({spaghetti})  frozenset({mineral water})  0.059725   
50    frozenset({ground meat})  frozenset({mineral water})  0.040928   

    confidence      lift  leverage  
53    0.225115  2.291162  0.022088  
52    0.398915  2.291162  0.022088  
64    0.250559  1.439085  0.018223  
63    0.343032  1.439085  0.018223  
50    0.416554  1.747522  0.017507  


In [6]:
# Sort rules according to lift
top_lift = rules.sort_values(
                by='lift',
                ascending=False)

# Top 5 rules
print(top_lift[
    ['antecedents',
     'consequents',
     'support',
     'confidence',
     'lift']
].head())

                 antecedents                     consequents   support  \
53    frozenset({spaghetti})        frozenset({ground meat})  0.039195   
52  frozenset({ground meat})          frozenset({spaghetti})  0.039195   
67    frozenset({olive oil})          frozenset({spaghetti})  0.022930   
62         frozenset({soup})      frozenset({mineral water})  0.023064   
41         frozenset({milk})  frozenset({frozen vegetables})  0.023597   

    confidence      lift  
53    0.225115  2.291162  
52    0.398915  2.291162  
67    0.348178  1.999758  
62    0.456464  1.914955  
41    0.182099  1.910382  


In [7]:
# Sort according to Zhang's metric
top_zhang = rules.sort_values(
                by='zhangs_metric',
                ascending=False)

# Top 2 rules
print(top_zhang[
    ['antecedents',
     'consequents',
     'zhangs_metric']
].head(2))

                 antecedents               consequents  zhangs_metric
53    frozenset({spaghetti})  frozenset({ground meat})       0.682343
52  frozenset({ground meat})    frozenset({spaghetti})       0.624943


In [8]:
# Sort in ascending order
bottom_zhang = rules.sort_values(
                    by='zhangs_metric',
                    ascending=True)

# Bottom 2 rules
print(bottom_zhang[
    ['antecedents',
     'consequents',
     'zhangs_metric']
].head(2))

                  antecedents                 consequents  zhangs_metric
35  frozenset({french fries})  frozenset({mineral water})      -0.200452
38     frozenset({spaghetti})   frozenset({french fries})      -0.086602
